# For this version of the reduction, I took 10%-40% of the FT as the depth range

# Results: 5% NASA ARC PLA on Glass has been determined to have the following mechanical properties:
- Young's Modulus: 18.9966 ± 0.0667 GPa
- Hardness: 1.4784 ± 0.0067 GPa

In [3]:
# Imports necessary for data analysis and reduction

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Functions

## Data Reducer

In [6]:
# DATA FRAME CREATION AND VISUALIZER

def tholin_analyzer(tholin_raw_data):
    
    # Creating new and reduced data frame
    tholin_raw_data = tholin_raw_data[['DEPTH', 'FILM MODULUS', 'HARDNESS']]
    tholin_raw_data = tholin_raw_data.dropna()
    tholin_raw_data.set_index('DEPTH', inplace=True)
    tholin_raw_data = tholin_raw_data.drop('nm', axis=0)
    tholin_raw_data = tholin_raw_data.astype(float)

    # If figure is desired, make the size figsize=(12, 4)
    
    return tholin_raw_data

## Intermediate Calculations

In [8]:
# DEPTH RANGE DETERMINANT AND MECHANICAL PROPERTY CALCULATIONS

def mechanical_property_calculations(film_thickness, calculation_data):
    # Making the average values and corresponding standard deviations global for final calculations
    global start, end, avg_e, avg_e_std, avg_h, avg_h_std, e_sample, e_std_sample, h_sample, h_std_sample
    
    # Defining the depth range to extract data from; 10%-40% of the film thickness
    start = film_thickness * 0.10
    end = film_thickness * 0.40

    # Creating empty modulus and hardness arrays to then append to and do calculations with
    modulus_set = np.array([])
    hardness_set = np.array([])

    # Young's modulus calculations
    for element in calculation_data.index:
        if ((element > start and element < end) == True):
            modulus_set = np.append(modulus_set, calculation_data.loc[element, 'FILM MODULUS'])
    avg_e = modulus_set.mean()
    avg_e_std = modulus_set.std()

    # Hardness calculations
    for element in calculation_data.index:
        if ((element > start and element < end) == True):
            hardness_set = np.append(hardness_set, calculation_data.loc[element, 'HARDNESS'])
    avg_h = hardness_set.mean()
    avg_h_std = hardness_set.std()

    e_sample = np.append(e_sample, avg_e)
    e_std_sample = np.append(e_std_sample, avg_e_std)
    h_sample = np.append(h_sample, avg_h)
    h_std_sample = np.append(h_std_sample, avg_h_std)
    
    return

## Display Current Sample Values

In [10]:
def sample_values():
    print(f'Defined film thickness: {film_thickness} nm')
    print(f'Selected depth range for calculations (10%-40% of film thickness): [{start:.2f}, {end:.2f}]\n')
    print(f'E Sample: {e_sample}\n')
    print(f'E Std Sample: {e_std_sample}\n')
    print(f'H Sample: {h_sample}\n')
    print(f'H Std Sample: {h_std_sample}\n')

    print(f'Length of E Sample array: {len(e_sample)}')
    print(f'Length of E Std Sample array: {len(e_std_sample)}')
    print(f'Length of H Sample array: {len(h_sample)}')
    print(f'Length of H Std Sample array: {len(h_std_sample)}\n')
    
    return

## Weighted Means & Standard Deviations

In [12]:
def final_weighted_values(e_sample, e_std_sample, h_sample, h_std_sample):
    
    # Creation of the data frame
    df = pd.DataFrame({
    'E$_i$': e_sample,
    '\u03C3$_{E,i}$': e_std_sample,
    'H$_i$': h_sample,
    '\u03C3$_{H,i}$': h_std_sample
    })

    # Weighted values calculations

    # Weighted average E
    numerator = (df['E$_i$'] * (1 / (df['\u03C3$_{E,i}$'] ** 2))).sum()
    denominator = (1 / (df['\u03C3$_{E,i}$'] ** 2)).sum()
    weighted_avg_e = numerator / denominator

    # Weighted E standard deviation
    weighted_std_e = (1 / (1 / (df['\u03C3$_{E,i}$'] ** 2)).sum()) ** (1/2)
    
    # Weighted average H
    numerator = (df['H$_i$'] * (1 / (df['\u03C3$_{H,i}$'] ** 2))).sum()
    denominator = (1 / (df['\u03C3$_{H,i}$'] ** 2)).sum()
    weighted_avg_h = numerator / denominator
    
    # Weighted H standard deviation
    weighted_std_h = (1 / (1 / (df['\u03C3$_{H,i}$'] ** 2)).sum()) ** (1/2)

    # Print final calculations for your tholin sample
    print('The final results for your sample are:\n')
    print(f'Young\'s Modulus: {weighted_avg_e:.4f} \u00B1 {weighted_std_e:.4f} GPa')
    print(f'Hardness: {weighted_avg_h:.4f} \u00B1 {weighted_std_h:.4f} GPa\n')
    print('Associated Data Frame is below.\n')

    return df

## Array Reset

In [14]:
def reset_values():
    global e_sample, e_std_sample, h_sample, h_std_sample
    
    e_sample = np.array([])
    e_std_sample = np.array([])
    h_sample = np.array([])
    h_std_sample = np.array([])
    print('''
e_sample = []
e_std_sample = []
h_sample = []
h_std_sample = []

All storage arrays have been reset to empty.
''')
    
    return

In [15]:
# Creating storage arrays for all values of this sample
# Reset to all of these arrays are at the bottom of this notebook

e_sample = np.array([])
e_std_sample = np.array([])
h_sample = np.array([])
h_std_sample = np.array([])

# Film Thickness Defining

In [318]:
film_thickness = float(input('Input the film thickness for this test (nm):'))

Input the film thickness for this test (nm): 639.54


# Data Reading & Reduction

In [320]:
data_folder = '/Users/ricardo/Yu Lab/Tholin Data Reduction/Data Excel Files/NASA ARC/5% Plasma Glass/'
data_file = input('Input the Excel file to read data from:')

Input the Excel file to read data from: 5% NASA ARC PLA Area 25 - skipped 24 - EDITED.xlsx


In [321]:
skip_sheets = input('Enter sheet numbers you wish to exclude:')
if skip_sheets == 'None':
    skip_sheets = []
else:
    skip_sheets = skip_sheets.split(', ')
    skip_sheets = [int(i) for i in skip_sheets]

Enter sheet numbers you wish to exclude: 1, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 15


In [322]:
# max_sheet = int(input('Enter the index of the last sheet in the Excel file:'))
max_sheet = 16

In [323]:
for sheet in np.arange(1, max_sheet + 1):
    if sheet in skip_sheets:
        continue
    tholin_raw_data = pd.read_excel(data_folder + data_file, sheet_name=f'Test {sheet}')
    calculation_data = tholin_analyzer(tholin_raw_data)
    mechanical_properties = mechanical_property_calculations(film_thickness, calculation_data)
    print(f'Sheet used for calculation: {sheet}')

if skip_sheets == []:
    print(f'Sheets skipped: None')
for sheet in skip_sheets:
        print(f'Sheet skipped: {sheet}')

print()   
sample_values()
final_weighted_values(e_sample, e_std_sample, h_sample, h_std_sample)

Sheet used for calculation: 2
Sheet used for calculation: 8
Sheet skipped: 1
Sheet skipped: 3
Sheet skipped: 4
Sheet skipped: 5
Sheet skipped: 6
Sheet skipped: 7
Sheet skipped: 9
Sheet skipped: 10
Sheet skipped: 11
Sheet skipped: 12
Sheet skipped: 13
Sheet skipped: 14
Sheet skipped: 15

Defined film thickness: 639.54 nm
Selected depth range for calculations (10%-40% of film thickness): [63.95, 255.82]

E Sample: [21.65961277 18.55729526 16.10875043 23.17745437 22.81725412 21.24663209
 21.74290763 23.35289772 23.69959733 22.64166705 22.03903535 14.29219539
 21.11700153 15.12647647 11.96997088 18.96137233 20.22184466 17.02682379
 18.23697207 16.76578964 13.04769115 17.85109217 20.91868048 15.36666394
 18.99019387 19.76026625 19.98568579 20.35212915 16.85168295 22.74348371
 21.024066   17.91036609 14.89847189 17.06800377 23.17934634 23.98102764
 18.09734951 23.31575507 18.90944507 18.58960303 20.72969019 14.80754899
 19.34001712 21.4458671  20.82599643 19.34369997 22.78333378 20.42402717


,E$_i$,"σ$_{E,i}$",H$_i$,"σ$_{H,i}$"
0,21.659613,1.777648,1.517229,0.259792
1,18.557295,2.143865,1.279717,0.289277
2,16.108750,1.834425,1.065300,0.202657
3,23.177454,1.777811,1.683447,0.220614
4,22.817254,1.924406,1.625112,0.240853
...,...,...,...,...
151,17.469242,1.071293,1.240989,0.071898
152,17.462663,0.626787,1.293935,0.106684
153,18.265045,0.370563,1.394899,0.065566
154,16.240980,0.536344,1.309791,0.110457


# Reset all of your storage arrays with the function below #

In [325]:
#reset_values()